# 03 Tables and Keys
Portfolio course, module 1 of 9. Rebuilding my SQL portfolio step by step, starting from the CSV data.

Dataset: the retail database from my BSc Business Analytics I assignment (WU, 2020): 9 linked tables
(addresses, stores, employees, orders, articles, costs, prices, depots, stock). The data is in `data\retail_csv\`.

Covers: **foreign keys** (`REFERENCES`), designing several linked tables, data types, loading CSV data.
New compared with lesson 01: tables that point to each other.

In [2]:
# turns on the %%sql cell magic
%load_ext sql
# opens data\retail.db, creates the file if it doesn't exist
%sql sqlite:///../data/retail.db

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


## 1. Why: one sheet vs. linked tables

The PMP Excel report had one big sheet per month. Every trade row repeated the asset name, the asset class, the ISIN.
When something changed, it had to be fixed in every row, and one row always got missed.

A database splits that sheet into **several small tables, where each fact is stored once**.
The asset name lives in one table. A trade only stores *which* asset, as a number. When the name is needed, it's looked up.

That lookup is what `XLOOKUP` does in Excel and `pd.merge` does in pandas. In SQL it's a `JOIN` (module 06).
This module builds the tables the JOINs will work on.

## 2. How: primary keys and foreign keys

Lesson 01 had one table with a `PRIMARY KEY`. Now there are nine, and they point to each other.

- **Primary key**: the column that identifies each row. The row's ID card. Two stores can't both be store 11.
- **Foreign key**: a column that stores *another table's* primary key. `employee.sid` says "works in store 11".
  The database checks that store 11 exists. That check is the point: no employee can work in a store that isn't there.

```
store                      employee
+-----+-----+              +-----+-------+-----+-----+
| sid | aid |              | eid | first | sid | aid |
+-----+-----+              +-----+-------+-----+-----+
| 11  | 500 |  <---------  | 81  | Diana | 11  | 505 |
| 22  | 501 |  <---------  | 83  | Sally | 22  | 507 |
+-----+-----+              +-----+-------+-----+-----+
 primary key                primary key  foreign key
```

| Idea | Excel | pandas |
|---|---|---|
| primary key | the lookup column in the second sheet, every value once | `df.set_index('sid')`, index must be unique |
| foreign key | the column you look up *with* in `XLOOKUP` | the `on=` column in `pd.merge` |
| foreign key check | `XLOOKUP` returning `#N/A` | `merge(..., validate=...)` / rows lost in an inner merge |

## 3. Syntax: REFERENCES

```sql
column_name  INTEGER  NOT NULL  REFERENCES other_table(other_column)
```

Read it as: "this column must be filled in, and its value must exist in `other_table.other_column`".
Without `NOT NULL` the column may be empty, but if it's filled, the value must exist.

Types you need (SQLite):

| Type | For | Example |
|---|---|---|
| `INTEGER` | whole numbers, IDs | 500, 11, 234 |
| `REAL` | decimal numbers | 31.87 |
| `TEXT` | text | 'Druckereistrasse' |
| `DATE` | dates, as text 'YYYY-MM-DD' | '2020-08-31' |

**Order matters:** a table can only point to a table that already exists. Create the parents first.

## 4. Example (a library, not my data)

```sql
CREATE TABLE author (
  author_id  INTEGER PRIMARY KEY,
  name       TEXT NOT NULL
);

CREATE TABLE book (
  book_id    INTEGER PRIMARY KEY,
  title      TEXT NOT NULL,
  author_id  INTEGER NOT NULL REFERENCES author(author_id),
  price_eur  REAL,
  published  DATE
);
```

`author` comes first because `book` points to it. Each book needs an author that exists in `author`.

## 5. Exercises

Write each `CREATE TABLE` in the cell under its checkpoint. The `DROP TABLE IF EXISTS` line is already there,
so you can re-run a cell after fixing it.

After checkpoint 3 and after checkpoint 4, run the loader cell. It fills your tables from the CSVs and checks every foreign key.
If it prints **PROBLEM**, read the message, fix your `CREATE TABLE`, re-run that cell, then re-run the loader.

Stuck for more than 5 minutes: open the hint under the checkpoint, one at a time.
When the loader says **Done**, paste your cells into the course chat for a check.

### Checkpoint 1: `address`

Every store, employee and depot has an address.

| Column | Rule |
|---|---|
| `aid` | whole number, identifies the address |
| `address_line` | text, must be filled in |

<details><summary>Hint 1</summary>

The ID column gets `INTEGER PRIMARY KEY`, the text column `TEXT NOT NULL`. Same as lesson 01.
</details>

In [4]:
%%sql
DROP TABLE IF EXISTS address;

CREATE TABLE address (
    aid INTEGER PRIMARY KEY,
    address_line TEXT NOT NULL);

Running query in 'sqlite:///../data/retail.db'

++
||
++
++

### Checkpoint 2: `store`

| Column | Rule |
|---|---|
| `sid` | whole number, identifies the store |
| `aid` | whole number, must be filled in, points to `address` |

<details><summary>Hint 1</summary>

`aid` is a foreign key. Look at how `book.author_id` points to `author` in the example.
</details>
<details><summary>Hint 2</summary>

Pattern: `column INTEGER NOT NULL REFERENCES other_table(other_column)`
</details>

In [7]:
%%sql
DROP TABLE IF EXISTS store;

CREATE TABLE store (
    sid INTEGER PRIMARY KEY,
    aid INTEGER NOT NULL REFERENCES address(aid));

Running query in 'sqlite:///../data/retail.db'

++
||
++
++

### Checkpoint 3: `employee`

| Column | Rule |
|---|---|
| `eid` | whole number, identifies the employee |
| `first_name` | text, must be filled in |
| `last_name` | text, must be filled in |
| `sid` | whole number, must be filled in, points to `store` (where they work) |
| `aid` | whole number, points to `address` (home address). One employee has none in the data |

<details><summary>Hint 1</summary>

Two foreign keys: `sid` to `store`, `aid` to `address`.
</details>
<details><summary>Hint 2</summary>

`aid` may be empty, so no `NOT NULL` on it.
</details>

In [10]:
%%sql
DROP TABLE IF EXISTS employee;

CREATE TABLE employee (
    eid INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    sid INTEGER NOT NULL REFERENCES store(sid),
    aid INTEGER REFERENCES address(aid));

Running query in 'sqlite:///../data/retail.db'

++
||
++
++

**Run the loader.** Expected: `address 9`, `store 5`, `employee 9` rows loaded, the other six skipped, then `Done.`

In [11]:
# provided: fills your tables from data\retail_csv\ and checks the foreign keys (don't edit)
from course_loader import load_retail
load_retail()

Loading retail.db

  address              9 rows loaded
  store                5 rows loaded
  employee             9 rows loaded
  orders           skipped (not created yet)
  article          skipped (not created yet)
  production_cost  skipped (not created yet)
  price            skipped (not created yet)
  depot            skipped (not created yet)
  in_stock         skipped (not created yet)

Done.


### Checkpoint 4: the other six tables

Same pattern, one cell each. Think about the order: each table only after the tables it points to.

| Table | Columns |
|---|---|
| `orders` | `oid` (ID), `order_date` (date, required), `sid` (required, → store), `quantity` (whole number, required) |
| `article` | `article_no` (ID), `oid` (required, → orders) |
| `production_cost` | `pcid` (ID), `article_no` (required, → article), `transportation_cost` (decimal, required), `material_cost` (decimal, required) |
| `price` | `pid` (ID), `article_no` (required, → article), `valid_from` (date, required), `price_eur` (decimal, required) |
| `depot` | `did` (ID), `aid` (required, → address) |
| `in_stock` | `stid` (ID), `did` (required, → depot), `article_no` (required, → article), `quantity` (whole number, required) |

<details><summary>Hint 1: order</summary>

Ask for each table: what does it point to? `in_stock` points to `depot` and `article`, so it comes last.
</details>
<details><summary>Hint 2: why "orders"</summary>

`ORDER` is a reserved word (as in `ORDER BY`), so the table is called `orders`.
</details>

In [24]:
%%sql
DROP TABLE IF EXISTS orders;

CREATE TABLE orders (
    oid INTEGER PRIMARY KEY,
    order_date DATE NOT NULL,
    sid INTEGER NOT NULL REFERENCES store(sid),
    quantity INTEGER NOT NULL
);

Running query in 'sqlite:///../data/retail.db'

++
||
++
++

In [16]:
%%sql
DROP TABLE IF EXISTS article;

CREATE TABLE article (
    article_no INTEGER PRIMARY KEY,
    oid INTEGER NOT NULL REFERENCES orders(oid));


Running query in 'sqlite:///../data/retail.db'

++
||
++
++

In [22]:
%%sql
DROP TABLE IF EXISTS production_cost;

CREATE TABLE production_cost (
    pcid INTEGER PRIMARY KEY,
    article_no INTEGER NOT NULL REFERENCES article(article_no),
    transportation_cost REAL NOT NULL,
    material_cost REAL NOT NULL
);

Running query in 'sqlite:///../data/retail.db'

++
||
++
++

In [18]:
%%sql
DROP TABLE IF EXISTS price;

CREATE TABLE price (
    pid INTEGER PRIMARY KEY,
    article_no INTEGER NOT NULL REFERENCES article(article_no),
    valid_from DATE NOT NULL,
    price_eur REAL NOT NULL
);

Running query in 'sqlite:///../data/retail.db'

++
||
++
++

In [19]:
%%sql
DROP TABLE IF EXISTS depot;

CREATE TABLE depot (
    did INTEGER PRIMARY KEY,
    aid INTEGER NOT NULL REFERENCES address(aid)
);

Running query in 'sqlite:///../data/retail.db'

++
||
++
++

In [20]:
%%sql
DROP TABLE IF EXISTS in_stock;

CREATE TABLE in_stock (
    stid INTEGER PRIMARY KEY,
    did INTEGER NOT NULL REFERENCES depot(did),
    article_no INTEGER NOT NULL REFERENCES article(article_no),
    quantity INTEGER NOT NULL
);

Running query in 'sqlite:///../data/retail.db'

++
||
++
++

**Run the loader again.** Expected: all nine tables loaded, then `Done.`

In [23]:
from course_loader import load_retail
load_retail()

Loading retail.db

  address              9 rows loaded
  store                5 rows loaded
  employee             9 rows loaded
  orders               9 rows loaded
  article              9 rows loaded
  production_cost      9 rows loaded
  price                9 rows loaded
  depot                2 rows loaded
  in_stock             6 rows loaded

Done.


### Checkpoint 5: look at the data

Show all employees. Question for the chat: which employee has no home address, and how does that show up?

In [36]:
%%sql

SELECT * FROM employee;

SELECT first_name, last_name FROM employee WHERE aid IS NULL;


Running query in 'sqlite:///../data/retail.db'

first_name,last_name
Benjamin,Whichaw


### Checkpoint 6: see the foreign key work

Try to add an employee who works in store 99, which doesn't exist. The loader turned foreign-key checks on for its own
connection, so turn them on here first:

```sql
PRAGMA foreign_keys = ON;
INSERT INTO employee (eid, first_name, last_name, sid) VALUES (90, 'Test', 'Person', 99);
```

Run it. You should get an error. Write down in one sentence what the error means, in the mistakes section below.

In [39]:
%%sql
PRAGMA foreign_keys = ON;
INSERT INTO employee (eid, first_name, last_name, sid) VALUES (90, 'Test', 'Person', 99);

Running query in 'sqlite:///../data/retail.db'

RuntimeError: (sqlite3.IntegrityError) FOREIGN KEY constraint failed
[SQL: INSERT INTO employee (eid, first_name, last_name, sid) VALUES (90, 'Test', 'Person', 99);]
(Background on this error at: https://sqlalche.me/e/21/gkpj)


In [40]:
%%sql
-- provided: turns the check off again in this notebook, so the DROP TABLE cells above can be re-run
PRAGMA foreign_keys = OFF;

Running query in 'sqlite:///../data/retail.db'

++
||
++
++

## 6. My mistakes

Learning points from this module (like in the project notebooks):

- **Column names must match the CSV exactly.** I wrote `address_id` / `address` instead of `aid` / `address_line`. The loader inserts by name, so it would have failed, and every later `REFERENCES address(aid)` would point to a column that doesn't exist.
- **`NOT NULL` and `REFERENCES` are two separate rules.** `REFERENCES` only checks values that are filled in: an empty cell passes. `NOT NULL` is what forbids empty cells. Employee 89 (Benjamin Whichaw) has no home address, so `employee.aid` can't be `NOT NULL`.
- **Don't overcorrect.** When I took `NOT NULL` off `aid`, I also took it off `sid`. Nothing broke, because the CSV has no employee without a store, so this kind of mistake stays silent until bad data arrives. Decide per column.
- **Copied cells keep the old foreign key.** I built `production_cost` from the `article` cell and left `oid REFERENCES orders(oid)` in. A cost belongs to an article: `article_no REFERENCES article(article_no)`. The loader caught it with a clear message.
- **Creation order follows the arrows.** A table can only point to a table that already exists: `address` → `store` → `employee`, `orders` → `article` → `production_cost` / `price` / `in_stock`, `depot` before `in_stock`.
- **Use the same type for the same kind of data.** I first had `order_date TEXT` but `valid_from DATE`. SQLite stores both the same way, but the schema should be consistent.
- **`IS NULL`, not `= NULL`.** Nothing equals NULL, not even NULL, so `aid = NULL` returns no rows.
- **`FOREIGN KEY constraint failed`** means SQLite looked up the value in the parent table, didn't find it (store 99), and refused the row. It only checks when `PRAGMA foreign_keys = ON`, because SQLite has the check off by default.

## 7. Key words for the cheat sheet

Added to `00_key_words.md` under **03 Tables and Keys**:

| Key word | Does | Section |
|---|---|---|
| `REFERENCES table(column)` | foreign key: a filled-in value must exist in that column of the other table | [3. Syntax](03_tables_and_keys.ipynb#3-syntax-references) |
| foreign key | column that points to the primary key of another table (like the lookup value in `XLOOKUP`) | [2. How](03_tables_and_keys.ipynb#2-how-primary-keys-and-foreign-keys) |
| `INTEGER` / `REAL` / `TEXT` / `DATE` | whole number / decimal / text / date (SQLite stores dates as text like `'2020-03-15'`) | [3. Syntax](03_tables_and_keys.ipynb#3-syntax-references) |
| `PRAGMA foreign_keys = ON` | turns foreign key checks on for this connection, off by default in SQLite | [Checkpoint 6](03_tables_and_keys.ipynb#checkpoint-6-see-the-foreign-key-work) |